# BioRAG-Hybrid: implementation status notebook

This notebook records exactly what has been implemented in Evaluation 1, which data were used, which papers define the project, what algorithms and parameters are actually running, and what stack is recommended for the follow-up implementation.

> **Important:** the current runnable result is a synthetic smoke test. The official BioASQ11 JSON was inspected, but it does not contain embedded article text, so it has not been used to claim official retrieval performance.

**Project:** BioRAG-Hybrid — Classical–neural biomedical retrieval-augmented question answering  
**Current milestone:** transparent CPU baseline with hybrid retrieval, HMM/Viterbi evidence selection, and extractive answering

## 1. Executive summary

### Implemented now

1. BioASQ-shaped JSON loading and validation.
2. Biomedical-safe whitespace normalization, tokenization, sentence splitting, and character n-grams.
3. BM25 lexical retrieval.
4. A deterministic character 3-gram cosine signal as a lightweight semantic **proxy**.
5. Weighted hybrid ranking: 0.60 lexical + 0.40 character n-gram signal.
6. A fixed three-state HMM decoded with log-space Viterbi: `SUPPORT`, `CONTEXT`, `NOISE`.
7. Verbatim extractive evidence selection with a two-sentence output limit.
8. Recall@k, MRR, exact-match helper, token F1, and ROUGE-L metrics.
9. Unit/integration tests, an end-to-end CLI, dataset inspection, editable diagrams, and presentation artifacts.

### Not implemented yet

- A learned biomedical dense encoder such as MedCPT or BioBERT.
- CBOW/Skip-gram training.
- A neural reranker or cross-encoder.
- An RNN/GRU summarizer.
- Official BioASQ retrieval/answer scores on a searchable article-text corpus.
- Full negation, conflict, entailment, and expert supported-claim evaluation.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys
from collections import Counter

# Make the notebook work when started from the repository root or notebooks/.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'BioRAG-Hybrid-Evaluation1']
ROOT = next(path for path in candidates if (path / 'src' / 'biorag').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from biorag.data import load_questions
from biorag.hmm import EvidenceHMM
from biorag.retrieval import HybridRetriever
from biorag.run_pipeline import run

print('Project root:', ROOT)
print('Python:', sys.version.split()[0])
print('Implementation package:', SRC / 'biorag')

: 

## 2. Reproducibility configuration

The configuration below is the exact Evaluation-1 prototype configuration. The code itself uses BM25 constants `k1=1.5` and `b=0.75`, while the project configuration controls the fusion weights, document cutoff, HMM thresholds, and answer length.

In [ ]:
config_path = ROOT / 'config' / 'prototype.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
print(json.dumps(config, indent=2))

### Exact current method

For each query and document, the current prototype computes:

```text
lexical = BM25(query, document),             k1 = 1.5, b = 0.75
proxy    = cosine(character_3grams(query), character_3grams(document))
score    = 0.60 * minmax(lexical) + 0.40 * minmax(proxy)
```

The HMM then computes question-token overlap for each sentence. Overlap `>= 0.18` is observed as `HIGH`, overlap `>= 0.06` as `MID`, and lower overlap as `LOW`. Viterbi chooses the most probable state sequence using hand-specified start, transition, and emission probabilities. This is a transparent prototype; it is not a trained HMM and `SUPPORT` is not an entailment certificate.

## 3. Datasets used

### A. Included synthetic smoke fixture — used for the runnable result

- File: `data/sample/bioasq_sample.json`
- Six questions covering summary, yes/no, and factoid styles.
- Thirteen small synthetic documents with article text.
- BioASQ-shaped fields: question ID, question type, documents, snippets, exact answers, and ideal answers.
- Purpose: verify that the complete loader → retrieval → evidence → answer → metrics path executes without external downloads.

### B. Official BioASQ11 training JSON — used for inspection only

- File: `data/official/training11b.json`
- Source: [BioASQ-QA Zenodo record](https://doi.org/10.5281/zenodo.7655130) and the [BioASQ data portal](https://participants-area.bioasq.org/datasets/).
- Local records: 4,719. The 2023 descriptor reports 4,721 questions at publication time; these are different release/count statements.
- Question types: factoid 1,417; list 901; summary 1,130; yes/no 1,271.
- Document references: 42,608; snippets: 56,795; embedded article texts: 0.
- SHA-256: `6df656862ca860efc355c7805d07ddca700d64ecc3785c519a49afccaaeeac98`.
- Therefore, it cannot be used as the retrieval corpus by itself. A separate, versioned, leakage-safe article/abstract text corpus must be aligned to the BioASQ document identifiers.

**No official BioASQ accuracy claim is made in this notebook.**

In [ ]:
sample_path = ROOT / 'data' / 'sample' / 'bioasq_sample.json'
official_path = ROOT / 'data' / 'official' / 'training11b.json'
sample_questions = load_questions(sample_path)
official_questions = load_questions(official_path)

sample_documents = {doc.doc_id for question in sample_questions for doc in question.documents}
embedded_official_texts = sum(
    bool(document.text.strip())
    for question in official_questions
    for document in question.documents
)
official_types = Counter(question.question_type for question in official_questions)

print('Synthetic fixture')
print('  questions:', len(sample_questions))
print('  unique documents:', len(sample_documents))
print('  SHA-256:', hashlib.sha256(sample_path.read_bytes()).hexdigest())
print('  question types:', Counter(question.question_type for question in sample_questions))
print()
print('Official BioASQ11 inspection')
print('  questions:', len(official_questions))
print('  question types:', dict(sorted(official_types.items())))
print('  embedded article-text fields:', embedded_official_texts)
print('  SHA-256:', hashlib.sha256(official_path.read_bytes()).hexdigest())

## 4. Base paper and supporting papers

### Recommended base methodology paper

**Xiong, Jin, Lu, and Zhang (2024), _Benchmarking Retrieval-Augmented Generation for Medicine_, Findings of ACL 2024.**  [ACL Anthology](https://aclanthology.org/2024.findings-acl.372/) · DOI: `10.18653/v1/2024.findings-acl.372` · [MedRAG code](https://github.com/gzxiong/MedRAG)

This is the best methodological base paper for the follow-up because it directly compares BM25, dense/biomedical retrieval, and rank fusion in medical RAG. It motivates our BM25-versus-semantic-versus-hybrid ablation. We are **not** claiming to reproduce its MIRAGE scale, large corpora, LLMs, or reported scores.

### Dataset/benchmark paper

**Krithara, Nentidis, Bougiatiotis, and Paliouras (2023), _BioASQ-QA: A manually curated corpus for Biomedical Question Answering_, Scientific Data.**  [Nature](https://www.nature.com/articles/s41597-023-02068-4) · DOI: `10.1038/s41597-023-02068-4`

This paper defines the target question types, supporting article/snippet annotations, exact answers, ideal paragraph answers, and the expert-curated nature of BioASQ.

### Reliability-oriented follow-up paper

**Sohn et al. (2025), _Rationale-Guided Retrieval Augmented Generation for Medical Question Answering_, NAACL 2025.**  [ACL Anthology](https://aclanthology.org/2025.naacl-long.635/) · DOI: `10.18653/v1/2025.naacl-long.635` · [RAG² code](https://github.com/dmis-lab/RAG2)

This is a later reliability reference for rationale-guided retrieval, balanced multi-corpus retrieval, MedCPT reranking, and learned evidence filtering. Its large corpus and H100-scale filter training are future-reference ideas, not components currently implemented here.

## 5. What exact implementation is running?

| Stage | Exact implementation now | File | Status |
|---|---|---|---|
| Data loading | BioASQ-shaped JSON loader; preserves documents, snippets, exact answers, ideal answers | `src/biorag/data.py` | Implemented |
| Normalization | Lowercase, whitespace cleanup, en-dash/em-dash normalization, `µ` → `u` | `src/biorag/text.py` | Implemented |
| Tokenization | Regex tokens with a small stopword list | `src/biorag/text.py` | Implemented |
| Lexical retrieval | BM25, `k1=1.5`, `b=0.75` | `src/biorag/retrieval.py` | Implemented |
| Semantic signal | Boundary-aware character 3-gram cosine similarity | `src/biorag/text.py`, `retrieval.py` | Proxy only |
| Fusion | Min-max normalize both scores, then `0.60 × BM25 + 0.40 × proxy` | `src/biorag/retrieval.py` | Implemented |
| Evidence selection | 3-state fixed HMM + log-space Viterbi | `src/biorag/hmm.py` | Implemented |
| Answering | Select up to two non-duplicate evidence sentences verbatim | `src/biorag/run_pipeline.py` | Implemented |
| Evaluation | Recall@3, MRR@3, token F1, ROUGE-L; exact-match helper exists | `src/biorag/metrics.py` | Implemented |

### Exact answer to the model question

A proper pretrained biomedical model was **not used yet**. The current semantic component is deliberately a standard-library character n-gram proxy. Do not write that the current experiment used BioBERT, MedCPT, CBOW, Skip-gram, or an RNN.

In [ ]:
retriever = HybridRetriever([], lexical_weight=0.60, semantic_weight=0.40)
hmm = EvidenceHMM(support_threshold=0.18, context_threshold=0.06)

print('Retriever weights:', retriever.lexical_weight, retriever.semantic_weight)
print('BM25 constants: k1=1.5, b=0.75 (defined in HybridRetriever._bm25)')
print('HMM states:', hmm.states)
print('HMM observations:', hmm.observations)
print('HMM thresholds:', {'support': hmm.support_threshold, 'context': hmm.context_threshold})
print('HMM start probabilities:', hmm.start)
print('HMM transition probabilities:', hmm.transition)
print('HMM emission probabilities:', hmm.emission)
print('Document cutoff:', config['retrieval']['top_k_documents'])
print('Maximum selected sentences:', config['answer']['max_sentences'])

## 6. Execute the implemented pipeline

This cell runs the same standard-library pipeline used to create `artifacts/results/demo_results.json`. It is a demonstration on the synthetic fixture, not an official BioASQ experiment.

In [ ]:
demo_output = ROOT / 'artifacts' / 'results' / 'notebook_demo_results.json'
demo = run(
    data_path=sample_path,
    output_path=demo_output,
    lexical_weight=0.60,
    selector='hmm',
)
print(json.dumps({
    'records': demo['record_count'],
    'corpus_documents': demo['corpus_count'],
    'lexical_weight': demo['lexical_weight'],
    'selector': demo['selector'],
    'aggregate_metrics': demo['aggregate_metrics'],
}, indent=2))
print('Saved:', demo_output)

In [ ]:
for record in demo['records'][:2]:
    print('Question:', record['question'])
    print('Type:', record['question_type'])
    print('Top documents:', [item['id'] for item in record['ranked_documents']])
    print('Selected evidence:')
    for item in record['selected_evidence']:
        print(f"  [{item['state']}] {item['sentence']}")
    print('Predicted extractive summary:', record['predicted_summary'])
    print('Metrics:', record['metrics'])
    print('-' * 80)

### Observed smoke-test result

| Metric | Result on six-record synthetic fixture |
|---|---:|
| Records | 6 |
| Corpus documents | 13 |
| Recall@3 | 1.000000 |
| MRR@3 | 1.000000 |
| Summary token F1 | 0.443306 |
| ROUGE-L F1 | 0.429417 |

These values only demonstrate that the code runs on the included fixture. The fixture is synthetic and small, so these numbers must not be reported as BioASQ performance.

## 7. Recommended follow-up implementation for the team

### Step 1 — obtain the right retrieval corpus

Use the authorized BioASQ release for questions and labels, then build a separate searchable corpus of the referenced PubMed/MEDLINE abstracts. Store a version/date/checksum and align documents by PMID or stable BioASQ document identifier. Do not index the gold snippets as a replacement for article text. Split by question ID before fitting retrieval statistics, embeddings, thresholds, or a summarizer.

### Step 2 — replace the proxy with an exact biomedical encoder

**Recommended primary model:**

- Query encoder: `ncbi/MedCPT-Query-Encoder`
- Article encoder: `ncbi/MedCPT-Article-Encoder`
- Vector similarity: normalized dot product / cosine similarity
- First index: FAISS `IndexFlatIP` for an exact, reproducible small-corpus baseline
- Optional second-stage reranker: `ncbi/MedCPT-Cross-Encoder`

Keep the current BM25 baseline. Compare BM25-only, MedCPT-only, and hybrid BM25 + MedCPT. Start with weighted score fusion using validation-tuned weights; add reciprocal-rank fusion as a separate ablation because the MedRAG paper uses rank fusion. Pin `torch`, `transformers`, `faiss-cpu`, and model revisions in the experiment log.

### Step 3 — evidence selection

Keep the current HMM/Viterbi selector as the transparent baseline. Compare it with: (a) top-overlap sentence selection, (b) MedCPT sentence scoring, and (c) the optional MedCPT cross-encoder. Tune HMM thresholds only on validation data. Report gold-snippet recall and a manual supported-claim sample; do not treat the `SUPPORT` label as proof of entailment.

### Step 4 — answer generation

For a modern biomedical summarization extension, the practical recommendation is `razent/SciFive-base-Pubmed_PMC`, fine-tuned only after evidence retrieval is stable. If the course specifically requires an RNN, implement a GRU encoder-decoder as a controlled baseline rather than calling the current extractive stage an RNN. A reasonable reproducible GRU starting point is a 2-layer 256-dimensional encoder, 1-layer 256-dimensional attention decoder, dropout 0.1, teacher forcing during training, and early stopping on validation ROUGE-L. Record this as a new experiment; it is not part of the current implementation.

### Recommended order

`BioASQ-aligned abstracts → BM25 baseline → MedCPT dense retrieval → BM25/MedCPT fusion → HMM versus reranker evidence ablation → SciFive or GRU summary → supported-claim and negation analysis`

## 8. Follow-up experiment matrix

| Experiment | Retriever | Evidence selector | Output | Main metrics |
|---|---|---|---|---|
| B0 | BM25 | Top-overlap sentence | Extractive | Recall@k, MRR, snippet recall, F1, ROUGE-L |
| B1 | BM25 | HMM/Viterbi | Extractive | Same metrics + supported-claim review |
| B2 | MedCPT | Top score | Extractive | Same metrics |
| B3 | BM25 + MedCPT weighted fusion | HMM/Viterbi | Extractive | Same metrics |
| B4 | BM25 + MedCPT fusion | MedCPT cross-encoder reranking | Extractive | Same metrics + latency |
| G1 | Best retrieval system | Best evidence selector | SciFive summary | ROUGE-L, information precision/recall, readability |
| G2 | Best retrieval system | Best evidence selector | GRU summary, if required | Same summary metrics + training cost |

The minimum defensible next experiment is B0–B3 on a leakage-safe BioASQ split. Only after those results are stable should the team train G1 or G2.

## 9. Files and commands

### Main files

- `src/biorag/data.py` — loader and BioASQ-shaped records
- `src/biorag/text.py` — normalization, tokenization, sentences, n-grams
- `src/biorag/retrieval.py` — BM25 and hybrid ranking
- `src/biorag/hmm.py` — HMM/Viterbi evidence selection
- `src/biorag/metrics.py` — evaluation helpers
- `src/biorag/run_pipeline.py` — end-to-end CLI
- `data/sample/bioasq_sample.json` — synthetic fixture used for results
- `data/official/training11b.json` — BioASQ11 file inspected for schema/counts
- `artifacts/results/demo_results.json` — saved synthetic run
- `docs/02_literature_review.md` — verified paper review
- `docs/03_dataset_methodology.md` — dataset and future methodology

### Re-run from the repository root

```bash
PYTHONPATH=src python3 -m unittest discover -s tests -v
PYTHONPATH=src python3 -m biorag.run_pipeline \
  --data data/sample/bioasq_sample.json \
  --output artifacts/results/demo_results.json
```

### Final implementation boundary

The current work is a reproducible classical baseline and evidence-selection prototype. The follow-up team should add the searchable BioASQ-aligned corpus and MedCPT before making any claim about biomedical semantic retrieval or official BioASQ performance.